In [4]:
import numpy as np
import pymongo
from sklearn.decomposition import PCA
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["papers"]
paras = db["paragraphs"]

In [ ]:
def plot_embedding_pcas(field_name, colour_by='synthesis'): 

    # Function to insert line breaks for text wrapping
    def wrap_text(text, n=50):
        return "<br>".join([text[i:i+n] for i in range(0, len(text), n)])

    # Query MongoDB with conditions
    cursor = paras.find(
        {
            "$and": [
                {f"{field_name}": {"$exists": True}},
                {"doctype": "fla"},
                {"manually_classified": True}
            ]
        }
    )

    embeddings = []
    labels = []
    texts = []
    ids = []

    for doc in cursor:
        embedding = doc.get(f"{field_name}", [])
        if isinstance(embedding, list) and len(embedding) > 0:
            embeddings.append(embedding)
            labels.append(doc.get(f"{colour_by}", "Unknown"))  # Default if missing
            texts.append(wrap_text(doc.get("text", "No text available"), n=50))
            ids.append(str(doc["_id"]))  # Convert _id to string for Plotly
    embeddings = np.array(embeddings)

    if embeddings.ndim != 2:
        raise ValueError(f"Expected 2D embeddings array, but got shape {embeddings.shape}")

    pca = PCA(n_components=3)
    reduced_embeddings = pca.fit_transform(embeddings)
    df = pd.DataFrame(reduced_embeddings, columns=["PC1", "PC2", "PC3"])
    df[f"{colour_by}"] = labels
    df["text"] = texts
    df["_id"] = ids
    fig = px.scatter_3d(df, x="PC1", y="PC2", z="PC3", 
                        color=f"{colour_by}", 
                        title=f"3D PCA of {field_name} Embeddings",
                        labels={"PC1": "Principal Component 1", 
                                "PC2": "Principal Component 2", 
                                "PC3": "Principal Component 3"},
                        opacity=0.8,
                        hover_data={"text": True, "_id": True}
    )
    fig.update_layout(height=800)
    fig.update_traces(marker=dict(size=3))  

    fig_widget = go.FigureWidget(fig)

    clicked_ids = []


    def store_clicked(trace, points, selector):
        for i in points.point_inds:
            clicked_id = df.iloc[i]["_id"]
            clicked_ids.append(clicked_id)
            print(f"Clicked _id: {clicked_id}")

    fig_widget.data[0].on_click(store_clicked)
    fig_widget.show()
    return pca

In [7]:
# Hover over data points for the text

for vec_field in ['matbert_uncased_cls', 'matbert_uncased_mean', 'scibert_uncased_cls', 'scibert_uncased_mean']:
    pca = plot_embedding_pcas(vec_field, colour_by='synthesis')
    print(vec_field, pca.explained_variance_ratio_)

matbert_uncased_cls [0.31732377 0.05856873 0.04672051]


matbert_uncased_mean [0.37709881 0.18927578 0.03619843]


scibert_uncased_cls [0.2194267  0.14066515 0.07838937]


scibert_uncased_mean [0.23012192 0.14610235 0.0674238 ]
